# A quantized convolution layer, from Python to FPGA

This notebook builds one int8 convolution layer as MLIR through the `mlir_laksa` Python bindings, then compiles it with LAKSA.

The layer reads a `1x32x32x8` int8 feature map in NHWC layout, convolves it with eight `3x3x8` filters (stride 1, no padding), and requantizes the `i32` accumulators back to int8 with a fused ReLU. The output is `1x30x30x8`.

The IR is built by hand, op by op, so every step of the layer is visible. It is then lowered in two independent ways:

* to LAKSA's `emithls` dialect and on to Vitis HLS C++, which is the hardware design;
* to upstream MLIR's `emitc` dialect and on to plain C, which is the scalar reference the board's output is checked against.

The last section does both with a single `ladle` call and also writes the build scripts and the program that runs on the board.

`mlir_laksa` packages the upstream MLIR Python bindings together with LAKSA's own dialects (such as `emithls`) and pass pipelines (`mlir_laksa.conversion`).

In [1]:
import numpy as np

In [2]:
from mlir_laksa.ir import (
    AffineExpr,
    AffineMap,
    Context,
    DenseElementsAttr,
    InsertionPoint,
    IntegerAttr,
    IntegerType,
    Location,
    Module,
    RankedTensorType,
    UnitAttr,
)
from mlir_laksa.dialects import arith, func, linalg, tensor, emithls
from mlir_laksa.passmanager import PassManager
import mlir_laksa.conversion as conversion

## `constants.npz`

The untrained parameters of the layer:

* `weights`: `8x3x3x8` int8 in FHWC layout (output channel, kernel row, kernel column, input channel), which is the filter layout `linalg.conv_2d_nhwc_fhwc_q` expects.
* `multipliers`: eight int32 values, one per output channel. Together with a shift, each one encodes that channel's requantization scale as the fixed-point number `multiplier / 2^shift`.

The remaining parameters (zero points, shift, bias) are the same for every channel and are written inline further down.

In [3]:
CONSTANTS = np.load("constants.npz")
WEIGHTS, MULTIPLIERS = CONSTANTS["weights"], CONSTANTS["multipliers"]

## Attribute helpers

Constant tensors are written in MLIR as dense elements attributes, e.g. `dense<[1, 2, 3]> : tensor<3xi32>`. Three small helpers keep the code below short:

* `dense` turns a numpy array into such an attribute, with the given element type.
* `splat` builds a tensor in which every element has the same value. It prints compactly as `dense<40>`, and the HLS pipeline later turns it back into a scalar.
* `constant` wraps an attribute in an `arith.constant` op, so that it becomes an SSA value other ops can use.

In [4]:
def dense(array, elem_type):
    """dense<...> : tensor<shape x elem_type>, from a numpy array."""
    return DenseElementsAttr.get(
        np.ascontiguousarray(array),
        type=RankedTensorType.get(list(array.shape), elem_type),
    )

def splat(shape, elem_type, value):
    """dense<value> : tensor<shape x elem_type>"""
    return DenseElementsAttr.get_splat(
        RankedTensorType.get(shape, elem_type), IntegerAttr.get(elem_type, value)
    )

def constant(attr):
    """arith.constant of a dense elements attribute."""
    return arith.ConstantOp(attr.type, attr)

## Context and location

Every MLIR object lives in a `Context`, and every op carries a `Location` that diagnostics point to. Here all ops get `loc(unknown)`.

In a script both would be entered with a `with` statement. A `with` block cannot span several notebook cells, though, so they are entered by hand with `__enter__()` and stay active for the rest of the session. The insertion points below use the same trick.

In [5]:
ctx = Context()
ctx.__enter__()

loc = Location.unknown()
loc.__enter__()

loc(unknown)

## Module

`Module.create()` makes an empty `builtin.module`, the top-level container of the IR.

An `InsertionPoint` decides where newly created ops go. Pointing it at the module body places the function created next inside the module.

In [6]:
module = Module.create()
ip_module = InsertionPoint(module.body)
ip_module.__enter__()

## Types and affine maps

All tensors use NHWC layout. Input and output are int8, while the convolution accumulates in `i32` (`acc_type`), wide enough for sums of int8 products. `i64` only appears inside the requantization arithmetic.

A `linalg.generic` runs over a loop nest, here the four output dimensions `(d0, d1, d2, d3)` = (N, H, W, C), and one affine map per operand says which element the operand contributes at each point:

* `map_full` is the identity `(d0, d1, d2, d3) -> (d0, d1, d2, d3)`, for operands that have the full output shape.
* `map_bias` is `(d0, d1, d2, d3) -> (d3)`, for per-channel operands such as the bias, the multipliers and the shifts. Only the channel index is used, so each value is broadcast over all pixels.

In [7]:
i8 = IntegerType.get_signless(8)
i32 = IntegerType.get_signless(32)
i64 = IntegerType.get_signless(64)

input_type = RankedTensorType.get([1, 32, 32, 8], i8)
output_type = RankedTensorType.get([1, 30, 30, 8], i8)
acc_type = RankedTensorType.get([1, 30, 30, 8], i32)

d0, d1, d2, d3 = (AffineExpr.get_dim(i) for i in range(4))
map_bias = AffineMap.get(4, 0, [d3])
map_full = AffineMap.get(4, 0, [d0, d1, d2, d3])

## `func.func @main`

The entry function takes the input feature map and returns the output feature map. It becomes the top of the hardware design, which the generated HLS C++ below calls `main_top`.

`add_entry_block()` creates the function body with one block argument per input (`arg0`). The insertion point then moves into that block, so every op from here on lands inside `@main`.

In [8]:
func_op = func.FuncOp("main", ([input_type], [output_type]))
block = func_op.add_entry_block()
(arg0,) = block.arguments

ip_body = InsertionPoint(block)
ip_body.__enter__()

print(func_op)

"func.func"() <{function_type = (tensor<1x32x32x8xi8>) -> tensor<1x30x30x8xi8>, sym_name = "main"}> ({
^bb0(%arg0: tensor<1x32x32x8xi8>):
}) : () -> ()


## Constants

The constants used by the three ops below:

* `c_neg128` and `c0` are the zero points of the input and the weights. `c_neg128` also serves as the output zero point and as the lower clamp bound.
* `c127` is the upper clamp bound, the largest int8 value.
* `multipliers` and `shifts` are the per-channel requantization parameters. The shift is 40 for every channel.
* `c_half`, `c_neg_half`, `c31` and `c1_i64` belong to the rounding in the requantization step.
* `weights` is the filter tensor, and `bias` is an all-zero per-channel bias.

They are created up front, at the top of the function, because the body of a `linalg.generic` may use values defined outside of it.

In [9]:
c127 = arith.ConstantOp(i32, 127)
c_neg_half = arith.ConstantOp(i64, -1073741824)
c_half = arith.ConstantOp(i64, 1073741824)
c31 = arith.ConstantOp(i32, 31)
c1_i64 = arith.ConstantOp(i64, 1)
shifts = constant(splat([8], i8, 40))
multipliers = constant(dense(MULTIPLIERS, i32))
c0 = arith.ConstantOp(i32, 0)
c_neg128 = arith.ConstantOp(i32, -128)
weights = constant(dense(WEIGHTS, i8))
bias = constant(splat([8], i32, 0))

## Fill

Linalg ops use destination-passing style. Each op takes its inputs as `ins` and the initial value of its result as `outs`. `tensor.empty` provides that initial tensor as a pure shape without any data.

This first `linalg.generic` initializes the `i32` accumulator with the bias. The input is read through `map_bias`, so for every pixel the body just yields the bias of the current channel. All four iterators are `parallel`, since the output elements do not depend on each other.

In [10]:
acc_init = tensor.EmptyOp([1, 30, 30, 8], i32)

fill = linalg.GenericOp(
    [acc_type],
    [bias.result],
    [acc_init.result],
    [map_bias, map_full],
    ["parallel"] * 4,
)
fill_block = fill.regions[0].blocks.append(i32, i32)
with InsertionPoint(fill_block):
    in_, _out = fill_block.arguments
    linalg.YieldOp([in_])

print(fill)

%12 = "linalg.generic"(%10, %11) <{indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>], iterator_types = [#linalg.iterator_type<parallel>, #linalg.iterator_type<parallel>, #linalg.iterator_type<parallel>, #linalg.iterator_type<parallel>], operandSegmentSizes = array<i32: 1, 1>}> ({
^bb0(%arg1: i32, %arg2: i32):
  "linalg.yield"(%arg1) : (i32) -> ()
}) : (tensor<8xi32>, tensor<1x30x30x8xi32>) -> tensor<1x30x30x8xi32>


## Quantized convolution

`linalg.conv_2d_nhwc_fhwc_q` is linalg's named op for quantized 2-D convolution. For every output element it accumulates

`acc[n, oh, ow, f] += (x[n, oh+kh, ow+kw, c] - x_zp) * (w[f, kh, kw, c] - w_zp)`

over `kh`, `kw` and `c`, starting from the bias-filled accumulator passed as `outs`. Here `x_zp = -128` and `w_zp = 0`. Stride and dilation are both 1 and there is no padding, so the 3x3 kernel shrinks 32x32 to 30x30.

When a named op is created through its Python class, its region is still empty. `fill_builtin_region` fills in the body that spells out the computation, which is what the printout below shows.

In [11]:
conv = linalg.Conv2DNhwcFhwcQOp(
    [acc_type],
    [arg0, weights.result, c_neg128.result, c0.result],
    [fill.results[0]],
    strides=dense(np.array([1, 1], dtype=np.int64), i64),
    dilations=dense(np.array([1, 1], dtype=np.int64), i64),
)
linalg.fill_builtin_region(conv.operation)

print(conv)

%13 = "linalg.conv_2d_nhwc_fhwc_q"(%arg0, %9, %8, %7, %12) <{dilations = dense<1> : tensor<2xi64>, operandSegmentSizes = array<i32: 4, 1>, strides = dense<1> : tensor<2xi64>}> ({
^bb0(%arg1: i8, %arg2: i8, %arg3: i32, %arg4: i32, %arg5: i32):
  %14 = "arith.extsi"(%arg1) : (i8) -> i32
  %15 = "arith.subi"(%14, %arg3) <{overflowFlags = #arith.overflow<none>}> : (i32, i32) -> i32
  %16 = "arith.extsi"(%arg2) : (i8) -> i32
  %17 = "arith.subi"(%16, %arg4) <{overflowFlags = #arith.overflow<none>}> : (i32, i32) -> i32
  %18 = "arith.muli"(%15, %17) <{overflowFlags = #arith.overflow<none>}> : (i32, i32) -> i32
  %19 = "arith.addi"(%arg5, %18) <{overflowFlags = #arith.overflow<none>}> : (i32, i32) -> i32
  "linalg.yield"(%19) : (i32) -> ()
}) {linalg.memoized_indexing_maps = [affine_map<(d0, d1, d2, d3, d4, d5, d6) -> (d0, d1 + d4, d2 + d5, d6)>, affine_map<(d0, d1, d2, d3, d4, d5, d6) -> (d3, d4, d5, d6)>, affine_map<(d0, d1, d2, d3, d4, d5, d6) -> ()>, affine_map<(d0, d1, d2, d3, d4, d5, d6

## ReLU and quantization

The last `linalg.generic` brings the `i32` accumulator back to int8. Per element it computes

`out = clamp(((acc * multiplier + round) >> shift) + out_zp, -128, 127)`

with `out_zp = -128` and the `multiplier` and `shift` of the element's channel, read through `map_bias`. The product is formed in `i64`, and `round = 1 << (shift - 1)` rounds the shifted result to the nearest integer. If `shift > 31`, another `±2^30`, whose sign follows `acc`, is added as well. This is the double rounding of TOSA's `rescale` op, and it is what the `cmpi`/`select` pairs in the body implement.

The ReLU needs no op of its own. With an output zero point of -128, the real value 0 is stored as -128, so clamping at -128 is exactly `max(x, 0)` in real terms. The same clamp also saturates the result to the int8 range.

In [12]:
out_init = tensor.EmptyOp([1, 30, 30, 8], i8)

relu = linalg.GenericOp(
    [output_type],
    [conv.results[0], multipliers.result, shifts.result],
    [out_init.result],
    [map_full, map_bias, map_bias, map_full],
    ["parallel"] * 4,
)
relu_block = relu.regions[0].blocks.append(i32, i32, i8, i8)
with InsertionPoint(relu_block):
    in_, in_mul, in_shift, _out = relu_block.arguments
    shift_i32 = arith.ExtUIOp(i32, in_shift)
    acc = arith.ExtSIOp(i64, in_)
    mul = arith.ExtSIOp(i64, in_mul)
    prod = arith.MulIOp(acc.result, mul.result)
    shift_i64 = arith.ExtUIOp(i64, in_shift)
    one_shl = arith.ShLIOp(c1_i64.result, shift_i64.result)
    round_ = arith.ShRUIOp(one_shl.result, c1_i64.result)
    rounded = arith.AddIOp(prod.result, round_.result)
    is_pos = arith.CmpIOp(arith.CmpIPredicate.sge, in_, c0.result)
    half = arith.SelectOp(is_pos.result, c_half.result, c_neg_half.result)
    biased = arith.AddIOp(half.result, rounded.result)
    wide = arith.CmpIOp(arith.CmpIPredicate.sgt, shift_i32.result, c31.result)
    picked = arith.SelectOp(wide.result, biased.result, rounded.result)
    shifted = arith.ShRSIOp(picked.result, shift_i64.result)
    trunc32 = arith.TruncIOp(i32, shifted.result)
    zp = arith.AddIOp(trunc32.result, c_neg128.result)
    clamped_lo = arith.MaxSIOp(zp.result, c_neg128.result)
    clamped = arith.MinSIOp(clamped_lo.result, c127.result)
    narrow = arith.TruncIOp(i8, clamped.result)
    linalg.YieldOp([narrow.result])

print(relu)

%15 = "linalg.generic"(%13, %6, %5, %14) <{indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>, affine_map<(d0, d1, d2, d3) -> (d3)>, affine_map<(d0, d1, d2, d3) -> (d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>], iterator_types = [#linalg.iterator_type<parallel>, #linalg.iterator_type<parallel>, #linalg.iterator_type<parallel>, #linalg.iterator_type<parallel>], operandSegmentSizes = array<i32: 3, 1>}> ({
^bb0(%arg1: i32, %arg2: i32, %arg3: i8, %arg4: i8):
  %16 = "arith.extui"(%arg3) : (i8) -> i32
  %17 = "arith.extsi"(%arg1) : (i32) -> i64
  %18 = "arith.extsi"(%arg2) : (i32) -> i64
  %19 = "arith.muli"(%17, %18) <{overflowFlags = #arith.overflow<none>}> : (i64, i64) -> i64
  %20 = "arith.extui"(%arg3) : (i8) -> i64
  %21 = "arith.shli"(%4, %20) <{overflowFlags = #arith.overflow<none>}> : (i64, i64) -> i64
  %22 = "arith.shrui"(%21, %4) : (i64, i64) -> i64
  %23 = "arith.addi"(%19, %22) <{overflowFlags = #arith.overflow<none>}> : (i64, i64) -> i64
  %24 = "arit

## Return

`func.return` hands the requantized tensor back as the function result. Exiting `ip_body` leaves the function body, so the insertion point is back at module level.

The finished module is the input to everything that follows, and it is kept twice:

* `linalg_ir` is a string snapshot. A pass manager rewrites a module in place, so each lowering below parses its own fresh copy from this string, and `module` stays untouched.
* `input.mlir` is the same IR on disk, which `ladle` reads in the last section.

In [13]:
func.ReturnOp([relu.results[0]])
ip_body.__exit__(None, None, None)

linalg_ir = str(module)
print(module)

# Save for later
with open("input.mlir", "w") as f:
    module.operation.print(file=f)

#map = affine_map<(d0, d1, d2, d3) -> (d3)>
#map1 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2, d3)>
module {
  func.func @main(%arg0: tensor<1x32x32x8xi8>) -> tensor<1x30x30x8xi8> {
    %c127_i32 = arith.constant 127 : i32
    %c-1073741824_i64 = arith.constant -1073741824 : i64
    %c1073741824_i64 = arith.constant 1073741824 : i64
    %c31_i32 = arith.constant 31 : i32
    %c1_i64 = arith.constant 1 : i64
    %cst = arith.constant dense<40> : tensor<8xi8>
    %cst_0 = arith.constant dense<[1288418624, 1300707578, 1298230215, 1293868576, 1310478957, 1307418510, 1313803263, 1306561467]> : tensor<8xi32>
    %c0_i32 = arith.constant 0 : i32
    %c-128_i32 = arith.constant -128 : i32
    %cst_1 = arith.constant dense<"0x3A971B7863DFAEAE6E9FA742A6D9AB301B052A3BF1413FB6EC4E28EFC60FF4B9650E1755841553970E4F814D52CBCCBD43C24EC473D98E2779CE48C0F54C31A48175AF3F539C32C3357144E98D85200A322FD4256F5559B979DB1308187F8B8DB08211798BFF403DD9ABD2E06EF6D5D9B1F28D2CFCCB236F81AA0EC1E03D00C6EE7AC44E00D96665

## To `emithls`

`emithls` is LAKSA's dialect for Vitis HLS C++. It models loops, arrays, streams and HLS pragmas as MLIR ops. The `convert-to-emithls` pipeline brings the `linalg` IR there in roughly four stages:

1. **Linalg preparation.** Named ops such as the convolution become `linalg.generic`, and splat constants such as the zero bias and the shift of 40 are replaced by scalars.
2. **Dataflow graph.** Each computation is outlined into its own function, and the graph is converted to LAKSA's `dfg` dialect, with one process per operator and channels in between.
3. **Loops.** The IR is bufferized, each process is lowered to loops that pull from and push to its channels, and the result is converted to `emithls`. For the convolution this is where a line buffer and a 3x3 sliding window appear, so every input pixel is read from the stream only once.
4. **HLS clean-up and pragmas.** Loops are fused, expressions folded, `+=`-style updates formed, and read/write functions added that move data between memory and the streams. Finally a Gurobi-backed design-space exploration decides where to put pragmas under a resource budget, and the IR is rewritten accordingly.

The budget comes from the pipeline's arguments `available_bram` and `available_dsp`. The defaults, 288 BRAM_18K and 1248 DSP, are the resources of the Kria K26 module on the KV260.

The printout is long; the C++ in the next section is easier to read.

In [14]:
hls_module = Module.parse(linalg_ir, context=ctx)

pm = PassManager(context=ctx)
conversion.add_convert_to_emithls_pipeline(pm, available_bram=288, available_dsp=1248)
pm.run(hls_module.operation)

print(hls_module)

module {
  emithls.include "algorithm"
  emithls.include "ap_int.h"
  emithls.include "cstddef"
  emithls.include "hls_stream.h"
  emithls.func @main_top_read_i8_0(%arg0: !emithls.ptr<i64>, %arg1: !emithls.array<8x!emithls.stream<i8>>) {
    emithls.pragma.inline off
    %const_index_0 = emithls.variable as const index = 1
    %const_index_1 = emithls.variable as const index = 8
    emithls.for %idx0 = 0 to 1024 step 1 {
      emithls.pragma.pipeline II=1 style=flp
      %0 = emithls.array.ptr_read %arg0[%idx0] : !emithls.ptr<i64> -> i64
      emithls.for %idx1 = 0 to 8 step 1 {
        %expr2 = emithls.expr : i8 {
          %expr0 = emithls.expr : index {
            %2 = emithls.arith.add %idx1, %const_index_0 : index
            %3 = emithls.arith.mul %2, %const_index_1 : index
            %4 = emithls.arith.sub %3, %const_index_0 : index
            emithls.yield %4 : index
          }
          %expr1 = emithls.expr : index {
            %2 = emithls.arith.mul %idx1, %const_index_

## To HLS C++

`emithls.translate_to_cpp` prints the `emithls` module as C++ for Vitis HLS, the same file `ladle` writes as `hw/main.cpp`. It is easiest to read from the bottom up:

* `main_top` is the top function. Its two `m_axi` ports are the input and output buffers in memory, and `#pragma HLS DATAFLOW` runs the four functions it calls concurrently, connected by `hls::stream` FIFOs.
* `main_top_read_i8_0` reads the input as 1024 64-bit words, one per pixel with its 8 int8 channels packed together, and splits them onto 8 streams. `main_top_write_i8_0` does the reverse for the 900 output pixels.
* `main_node_0` is the convolution. `array1` is a line buffer that keeps the last two input rows, and `array0` is the 3x3 window, so each input pixel is read from the stream once. The loop nest under the second `PIPELINE` pragma is fully unrolled, so all `8x3x3x8 = 576` products of one output pixel are computed in parallel. The bias fill has no node of its own: the bias is zero, so it has become the accumulator's initial value `var_tmp0 = 0`.
* `main_node_1` is the requantization. The shift is the constant 40, so the rounding term has been folded to `549755813888` (`2^39`) and the `shift > 31` test is gone. Only the multipliers still differ per channel, and they sit in a small ROM `cst0`.

Every multiplication carries `BIND_OP ... impl=dsp`, which binds it to a DSP explicitly and stops Vitis HLS from packing two multiplications into one DSP.

In [15]:
hls_cpp = emithls.translate_to_cpp(hls_module.operation)

print(hls_cpp)

#include "algorithm"
#include "ap_int.h"
#include "cstddef"
#include "hls_stream.h"
void main_top_read_i8_0(ap_int<64> *arg0, hls::stream<ap_int<8>> arg1[8])
{
  #pragma HLS INLINE off
  for (size_t idx0 = 0; idx0 < 1024; idx0 += 1) {
    #pragma HLS PIPELINE II=1 style=flp
    ap_int<64> elem_tmp0 = arg0[idx0];
    for (size_t idx1 = 0; idx1 < 8; idx1 += 1) {
      arg1[idx1].write(elem_tmp0.range((idx1 + 1) * 8 - 1, idx1 * 8));
    }
  }
}
void main_top_write_i8_0(hls::stream<ap_int<8>> arg0[8], ap_int<64> *arg1)
{
  #pragma HLS INLINE off
  for (size_t idx0 = 0; idx0 < 900; idx0 += 1) {
    #pragma HLS PIPELINE II=1 style=flp
    ap_int<64> var_tmp0 = 0;
    for (size_t idx1 = 0; idx1 < 8; idx1 += 1) {
      ap_int<8> data_tmp0 = arg0[idx1].read();
      var_tmp0.range((idx1 + 1) * 8 - 1, idx1 * 8) = data_tmp0;
    }
    arg1[idx0] = var_tmp0;
  }
}
void main_node_0(hls::stream<ap_int<8>> arg0[8], hls::stream<ap_int<32>> arg1[8])
{
  #pragma HLS INLINE off
  ap_int<8> array0[8][3][3

## To `emitc`

The second lowering starts again from `linalg_ir` and targets upstream MLIR's `emitc` dialect, which models plain C. None of the HLS restructuring happens here. The result is a straightforward scalar implementation that computes what the design is *meant* to compute, and it serves as the reference the board's output is compared against.

`add_convert_to_emitc_pipeline` adds the same passes as the `convert-to-laksa-emitc` pipeline that `ladle` runs:

* bufferize with plain, identity-layout memrefs, since C has no strided arrays;
* turn the returned tensor into an output parameter, since a C function cannot return an array;
* lower the linalg ops to loops and put the remaining buffer, the `i32` accumulator, on the stack;
* expand `maxsi`/`minsi` into compare and select, then convert everything to `emitc`.

In [16]:
emitc_module = Module.parse(linalg_ir, context=ctx)

pm = PassManager(context=ctx)
conversion.add_convert_to_emitc_pipeline(pm)
pm.run(emitc_module.operation)

print(emitc_module)

module {
  emitc.global static const @__constant_8xi32_0 : !emitc.array<8xi32> = dense<0>
  emitc.global static const @__constant_8x3x3x8xi8 : !emitc.array<8x3x3x8xi8> = dense<"0x3A971B7863DFAEAE6E9FA742A6D9AB301B052A3BF1413FB6EC4E28EFC60FF4B9650E1755841553970E4F814D52CBCCBD43C24EC473D98E2779CE48C0F54C31A48175AF3F539C32C3357144E98D85200A322FD4256F5559B979DB1308187F8B8DB08211798BFF403DD9ABD2E06EF6D5D9B1F28D2CFCCB236F81AA0EC1E03D00C6EE7AC44E00D966650E5B6BDEB4981B5D81F5758D3FB5F5768850CE45368D641D05CE09EC74E79BC06664E43B02C0ECE875D905EFEA9312668195A928174028C07857706CF608ED278BC5341EA7B7D8845AF989176D47750F90054CC0382EF3BE107E33812941AA57AD40B25D72370586299255D91ADF44961F6857A0213292992070342E1DF79DC4052E2CBAEA5D596A02918A19CB9C9AA0F46EF267E901FB5BF079B1FCE4015B7FB5142D5187009705F955FDB7DE90FEDF5482F75AADD055D4647BC2B0682A6420960F569EDF0D65843C9BBDEE111E1AE3D56D51B9F7AD7CDA0FB0EC68ED23A1EA75810211712A40C5E86194701DED4D079F8B0BFDCC02B8497BC4B768427FA666669BE915A4F6E844F49F6D743B4ADDDE6BA76

## To C

`translate_emitc_to_cpp` is the upstream EmitC translator. Its output is what `ladle` writes as `app/ref.h`.

The three loop nests are the fill, the convolution and the requantization, one after the other and unfused. Two details look odd but are deliberate:

* Additions, subtractions and multiplications go through unsigned types. MLIR integers wrap around on overflow, while signed overflow is undefined in C, so the lowering computes in unsigned arithmetic and casts back.
* Shifts are guarded as `amount < 64 ? x >> amount : 0`, because in C a shift by the full bit width or more is undefined.

In [17]:
c_src = conversion.translate_emitc_to_cpp(emitc_module.operation)

print(c_src)

static const int32_t __constant_8xi32_0[8] = {0, 0, 0, 0, 0, 0, 0, 0};
static const int8_t __constant_8x3x3x8xi8[8][3][3][8] = {58, -105, 27, 120, 99, -33, -82, -82, 110, -97, -89, 66, -90, -39, -85, 48, 27, 5, 42, 59, -15, 65, 63, -74, -20, 78, 40, -17, -58, 15, -12, -71, 101, 14, 23, 85, -124, 21, 83, -105, 14, 79, -127, 77, 82, -53, -52, -67, 67, -62, 78, -60, 115, -39, -114, 39, 121, -50, 72, -64, -11, 76, 49, -92, -127, 117, -81, 63, 83, -100, 50, -61, 53, 113, 68, -23, -115, -123, 32, 10, 50, 47, -44, 37, 111, 85, 89, -71, 121, -37, 19, 8, 24, 127, -117, -115, -80, -126, 17, 121, -117, -1, 64, 61, -39, -85, -46, -32, 110, -10, -43, -39, -79, -14, -115, 44, -4, -53, 35, 111, -127, -86, 14, -63, -32, 61, 0, -58, -18, 122, -60, 78, 0, -39, 102, 101, 14, 91, 107, -34, -76, -104, 27, 93, -127, -11, 117, -115, 63, -75, -11, 118, -120, 80, -50, 69, 54, -115, 100, 29, 5, -50, 9, -20, 116, -25, -101, -64, 102, 100, -28, 59, 2, -64, -20, -24, 117, -39, 5, -17, -22, -109, 18, 102, -127, -10

## One-liner

`ladle` is LAKSA's compiler driver, a thin wrapper around `laksa-opt` and `laksa-translate`. With `--hls` it runs both lowerings above on `input.mlir` and writes every artifact a board deployment needs into the directory given by `-o` (the current directory if omitted). The leading `!` hands the line to the shell.

```text
build/
├── hls.mlir            the design in emithls, as in "To emithls"
├── ref.mlir            the same input in emitc, as in "To emitc"
├── hw/                 for the build host
│   ├── main.cpp        Vitis HLS input, as in "To HLS C++"
│   ├── run_hls.tcl     C synthesis and IP export
│   ├── run_vivado.tcl  block design, synthesis, implementation, bitstream
│   └── build.sh        runs both, extracts main_top.bit
└── app/                for the board
    ├── app.h           buffer sizes and AXI-Lite register offsets
    ├── app.c           userspace driver, talks to /dev/laksa
    ├── app.dtsi        device tree overlay
    ├── ref.h           the scalar reference, as in "To C"
    ├── ref.c           compares the board's output against ref.h
    └── run.sh          loads the design, runs it, checks it
```

From here, run `hw/build.sh` on a machine with the Xilinx tools, copy the resulting `main_top.bit` into `app/`, and run `app/run.sh` on a Kria board with `laksa-hls-kria-driver` loaded. The repository README has the details.

In [18]:
!ladle input.mlir --hls --num-bram=288 --num-dsp=1248 -o build

INFO: Lowering input.mlir to hls.mlir through builtin.module(convert-to-emithls{available-bram=288 available-dsp=1248})...
INFO: Lowering input.mlir to ref.mlir through convert-to-laksa-emitc...
INFO: Writing hw/main.cpp from hls.mlir through emithls-to-cpp...
INFO: Writing hw/run_hls.tcl from hls.mlir through emithls-to-hls-tcl...
INFO: Writing hw/run_vivado.tcl from hls.mlir through emithls-to-vivado-tcl...
INFO: Writing hw/build.sh from hls.mlir through emithls-to-hls-build-script...
INFO: Writing app/app.h from hls.mlir through emithls-to-laksa-header...
INFO: Writing app/app.c from hls.mlir through emithls-to-laksa-app...
INFO: Writing app/app.dtsi from hls.mlir through emithls-to-kria-dtsi...
INFO: Writing app/ref.h from ref.mlir through mlir-to-cpp...
INFO: Writing app/ref.c from ref.mlir through emitc-to-laksa-ref...
INFO: Writing app/run.sh from hls.mlir through emithls-to-laksa-run-script...
INFO: Done, wrote 10 artifacts below 'build'
